# Tool choice

Tool use supports a parâmetro called `tool_choice` aquele allows you para specify como you want Claude para call tools. In este notebook, we'll take a look at como it works e quando para use it. antes going any further, Certifique-se you are comfortable com the basics of tool use com Claude.

quando working com the `tool_choice` parâmetro, we have three possible opções: 

* `auto` allows Claude para decide whether para call any provided tools ou não
* `tool` allows us para force Claude para sempre use a particular tool
* `any` tells Claude aquele it must use one of the provided tools, but doesn't force a particular tool

Let's take a look at each option in detail. We'll iniciar by importing the Anthropic SDK:

In [31]:
from anthropic import Anthropic
client = Anthropic()
MODEL_NAME = "claude-3-sonnet-20240229"

## Auto

Setting `tool_choice` para `auto` allows the model para automatically decide whether para use tools ou não.  este is the padrão behavior quando working com tools. 

para demonstrate este, we're going para provide Claude com a fake web buscar tool. We will ask Claude questions, alguns of qual would require calling the web buscar tool e other qual Claude should be able para answer on its own.

Let's iniciar by defining a tool called `web_search`.  Observe, para keep este demo simples, we're não actually busca the web aqui:

In [137]:
def web_search(topic):
    print(f"pretending to search the web for {topic}")

web_search_tool = {
    "name": "web_search",
    "description": "A tool to retrieve up to date information on a given topic by searching the web",
    "input_schema": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The topic to search the web for"
            },
        },
        "required": ["topic"]
    }
}


próximo, we escrever a função aquele accepts a user_query e passes it along para Claude, along com the `web_search_tool`. 

We also set `tool_choice` para `auto`:

```py
tool_choice={"type": "auto"}
```

aqui's the completo função:

In [145]:
from datetime import date

def chat_with_web_search(user_query):
    messages = [{"role": "user", "content": user_query}]

    system_prompt=f"""
    Answer as many questions as you can using your existing knowledge.  
    Only search the web for queries that you can not confidently answer.
    Today's date is {date.today().strftime("%B %d %Y")}
    If you think a user's question involves something in the future that hasn't happened yet, use the search tool.
    """

    response = client.messages.create(
        system=system_prompt,
        model=MODEL_NAME,
        messages=messages,
        max_tokens=1000,
        tool_choice={"type": "auto"},
        tools=[web_search_tool]
    )
    last_content_block = response.content[-1]
    if last_content_block.type == "text":
        print("Claude did NOT call a tool")
        print(f"Assistant: {last_content_block.text}")
    elif last_content_block.type == "tool_use":
        print("Claude wants to use a tool")
        print(last_content_block)

Let's iniciar com a question Claude should be able para answer sem using the tool:

In [139]:
chat_with_web_search("What color is the sky?")

Claude did NOT call a tool
Assistant: The sky appears blue during the day. This is because the Earth's atmosphere scatters more blue light from the sun than other colors, making the sky look blue.


quando we ask "o que cor is the sky?", Claude does não use the tool.  Let's try asking something aquele Claude should use the web buscar tool para answer:

In [140]:
chat_with_web_search("Who won the 2024 Miami Grand Prix?")

Claude wants to use a tool
ToolUseBlock(id='toolu_staging_018nwaaRebX33pHqoZZXDaSw', input={'topic': '2024 Miami Grand Prix winner'}, name='web_search', type='tool_use')


quando we ask "quem won the 2024 Miami Grand Prix?", Claude uses the web buscar tool! 

Let's try a poucos mais Exemplos:

In [141]:
# Claude should NOT need to use the tool for this:
chat_with_web_search("Who won the superbowl in 2022?")

Claude did NOT call a tool
Assistant: The Los Angeles Rams won Super Bowl LVI in 2022, defeating the Cincinnati Bengals by a score of 23-20. The game was played on February 13, 2022 at SoFi Stadium in Inglewood, California.


In [144]:
# Claude SHOULD use the tool for this:
chat_with_web_search("Who won the superbowl in 2024?")

Claude wants to use a tool
ToolUseBlock(id='toolu_staging_016XPwcprHAgYJBtN7A3jLhb', input={'topic': '2024 Super Bowl winner'}, name='web_search', type='tool_use')


### Your Prompt Matters!

quando working com `tool_choice` of `auto`, it's Importante aquele you spend tempo para escrever a detailed prompt.  frequentemente, Claude can be over-eager para call tools.  Writing a detailed prompt helps Claude determine quando para call a tool e quando não para.  In the acima example, we included specific instructions in the system prompt: 


```py
 system_prompt=f"""
    Answer as many questions as you can using your existing knowledge.  
    Only search the web for queries that you can not confidently answer.
    Today's date is {date.today().strftime("%B %d %Y")}
    If you think a user's question involves something in the future that hasn't happened yet, use the search tool.
"""
```



## Forcing a specific tool

We can force Claude para use a particular tool using `tool_choice`.  In the example abaixo, we've defined two simples tools: 
* `print_sentiment_scores` - a tool aquele "tricks" Claude into generating well-structured JSON saída containing sentiment analysis data.  For mais info on este approach, see [Extracting Structured JSON using Claude e Tool Use](https://github.com/anthropics/anthropic-cookbook/blob/principal/tool_use/extracting_structured_json.ipynb)
* `calculator` - a muito simples calculator tool aquele takes two numbers e adds them together 


In [111]:

tools = [
    {
        "name": "print_sentiment_scores",
        "description": "Prints the sentiment scores of a given tweet or piece of text.",
        "input_schema": {
            "type": "object",
            "properties": {
                "positive_score": {"type": "number", "description": "The positive sentiment score, ranging from 0.0 to 1.0."},
                "negative_score": {"type": "number", "description": "The negative sentiment score, ranging from 0.0 to 1.0."},
                "neutral_score": {"type": "number", "description": "The neutral sentiment score, ranging from 0.0 to 1.0."}
            },
            "required": ["positive_score", "negative_score", "neutral_score"]
        }
    },
    {
        "name": "calculator",
        "description": "Adds two number",
        "input_schema": {
            "type": "object",
            "properties": {
                "num1": {"type": "number", "description": "first number to add"},
                "num2": {"type": "number", "description": "second number to add"},
            },
            "required": ["num1", "num2"]
        }
    }
]

Our goal is para escrever a função called `analyze_tweet_sentiment` aquele takes a tweet e prints a basic sentiment analysis of aquele tweet.  eventualmente we will "force" Claude para use our sentiment analysis tool, but we'll iniciar by showing o que happens quando we **do não** force the tool use. 

In este primeiro "bad" versão of the `analyze_tweet_sentiment` função, we provide Claude com both tools. For the sake of comparison, we'll iniciar by setting tool_choice para "auto":

```py
tool_choice={"type": "auto"}
```

Observe aquele we are deliberately não providing Claude com a well-written prompt, para make it easier para Veja o impact of forcing the use of a particular tool.

In [124]:
def analyze_tweet_sentiment(query):
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "auto"},
        messages=[{"role": "user", "content": query}]
    )
    print(response)


Let's see o que happens quando we call the função com the tweet "Holy cow, I just made the maioria incredible meal!"

In [125]:
analyze_tweet_sentiment("Holy cow, I just made the most incredible meal!")

ToolsBetaMessage(id='msg_staging_01ApgXx7W7qsDugdaRWh6p21', content=[TextBlock(text="That's great to hear! I don't actually have the capability to assess sentiment from text, but it sounds like you're really excited and proud of the incredible meal you made. Cooking something delicious that you're proud of can definitely give a sense of accomplishment and happiness. Well done on creating such an amazing dish!", type='text')], model='claude-3-sonnet-20240229', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(input_tokens=429, output_tokens=69))


Claude does não call our sentiment analysis tool:
> "That's great to hear! I don't actually have the capability to assess sentiment from text, but it sounds like you're really excited and proud of the incredible meal you made

próximo, let's imagine someone tweets este: "I love my cats! I had four e just adopted 2 mais! Guess como muitos I have agora?"

In [128]:
analyze_tweet_sentiment("I love my cats! I had four and just adopted 2 more! Guess how many I have now?")

ToolsBetaMessage(id='msg_staging_018gTrwrx6YwBR2jjhdPooVg', content=[TextBlock(text="That's wonderful that you love your cats and adopted two more! To figure out how many cats you have now, I can use the calculator tool:", type='text'), ToolUseBlock(id='toolu_staging_01RFker5oMQoY6jErz5prmZg', input={'num1': 4, 'num2': 2}, name='calculator', type='tool_use')], model='claude-3-sonnet-20240229', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=442, output_tokens=101))


Claude wants para call the calculator tool:

> ToolUseBlock(id='toolu_staging_01RFker5oMQoY6jErz5prmZg', input={'num1': 4, 'num2': 2}, name='calculator', type='tool_use')

Clearly, este atual implementation is não doing o que we want (mostly because we set it up para fail). 

próximo, let's force Claude para **sempre** use the `print_sentiment_scores` tool by updating `tool_choice`:

```py
tool_choice={"type": "tool", "name": "print_sentiment_scores"}
```

In addition para setting `type` para `tool`, we must provide a particular tool name.

In [132]:
def analyze_tweet_sentiment(query):
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "tool", "name": "print_sentiment_scores"},
        messages=[{"role": "user", "content": query}]
    )
    print(response)

agora se we try prompting Claude com the same prompts de earlier, it's sempre going para call the `print_sentiment_scores` tool:

In [133]:
analyze_tweet_sentiment("Holy cow, I just made the most incredible meal!")

ToolsBetaMessage(id='msg_staging_018GtYk8Xvee3w8Eeh6pbgoq', content=[ToolUseBlock(id='toolu_staging_01FMRQ9pZniZqFUGQwTcFU4N', input={'positive_score': 0.9, 'negative_score': 0.0, 'neutral_score': 0.1}, name='print_sentiment_scores', type='tool_use')], model='claude-3-sonnet-20240229', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=527, output_tokens=79))


Claude calls our `print_sentiment_scores` tool:

> ToolUseBlock(id='toolu_staging_01FMRQ9pZniZqFUGQwTcFU4N', input={'positive_score': 0.9, 'negative_score': 0.0, 'neutral_score': 0.1}, name='print_sentiment_scores', type='tool_use')

Even se we try para trip up Claude com a "Math-y" tweet, it still sempre calls the `print_sentiment_scores` tool:

In [134]:
analyze_tweet_sentiment("I love my cats! I had four and just adopted 2 more! Guess how many I have now?")

ToolsBetaMessage(id='msg_staging_01RACamfrHdpvLxWaNwDfZEF', content=[ToolUseBlock(id='toolu_staging_01Wb6ZKSwKvqVSKLDAte9cKU', input={'positive_score': 0.8, 'negative_score': 0.0, 'neutral_score': 0.2}, name='print_sentiment_scores', type='tool_use')], model='claude-3-sonnet-20240229', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=540, output_tokens=79))


Even though we're forcing Claude para call our `print_sentiment_scores` tool, we should still employ alguns basic prompt engineering:

In [135]:
def analyze_tweet_sentiment(query):

    prompt = f"""
    Analyze the sentiment in the following tweet: 
    <tweet>{query}</tweet>
    """
    
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "auto"},
        messages=[{"role": "user", "content": prompt}]
    )
    print(response)

## Any

The final option for `tool_choice` is `any` qual allows us para tell Claude "you must call a tool, but you can pick qual one".  Imagine we want para criar a SMS chatbot using Claude.  The only way for este chatbot para actually "communicate" com a usuário is via SMS texto message. 

In the example abaixo, we make a muito simples texto-messaging assistant aquele has access para two tools:
* `send_text_to_user` sends a texto message para a usuário
* `get_customer_info` looks up customer data based on a username

The idea is para criar a chatbot aquele sempre calls one of estes tools e nunca responds com a non-tool resposta.  In todos situations, Claude should either respond atrás by trying para enviar a texto message ou calling `get_customer_info` para get mais customer information.

maioria importantly, we set `tool_choice` para "any":

```py
tool_choice={"type": "any"}
```

In [162]:
def send_text_to_user(text):
    # Sends a text to the user
    # We'll just print out the text to keep things simple:
    print(f"TEXT MESSAGE SENT: {text}")

def get_customer_info(username):
    return {
        "username": username,
        "email": f"{username}@email.com",
        "purchases": [
            {"id": 1, "product": "computer mouse"},
            {"id": 2, "product": "screen protector"},
            {"id": 3, "product": "usb charging cable"},
        ]
    }

tools = [
    {
        "name": "send_text_to_user",
        "description": "Sends a text message to a user",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The piece of text to be sent to the user via text message"},
            },
            "required": ["text"]
        }
    },
    {
        "name": "get_customer_info",
        "description": "gets information on a customer based on the customer's username.  Response includes email, username, and previous purchases. Only call this tool once a user has provided you with their username",
        "input_schema": {
            "type": "object",
            "properties": {
                "username": {"type": "string", "description": "The username of the user in question. "},
            },
            "required": ["username"]
        }
    },
]

system_prompt = """
All your communication with a user is done via text message.
Only call tools when you have enough information to accurately call them.  
Do not call the get_customer_info tool until a user has provided you with their username. This is important.
If you do not know a user's username, simply ask a user for their username.
"""

def sms_chatbot(user_message):
    messages = [{"role": "user", "content":user_message}]

    response = client.messages.create(
        system=system_prompt,
        model=MODEL_NAME,
        max_tokens=4096,
        tools=tools,
        tool_choice={"type": "any"},
        messages=messages
    )
    if response.stop_reason == "tool_use":
        last_content_block = response.content[-1]
        if last_content_block.type == 'tool_use':
            tool_name = last_content_block.name
            tool_inputs = last_content_block.input
            print(f"=======Claude Wants To Call The {tool_name} Tool=======")
            if tool_name == "send_text_to_user":
                send_text_to_user(tool_inputs["text"])
            elif tool_name == "get_customer_info":
                print(get_customer_info(tool_inputs["username"]))
            else:
                print("Oh dear, that tool doesn't exist!")
            
    else:
        print("No tool was called. This shouldn't happen!")
    

Let's iniciar simples:

In [163]:
sms_chatbot("Hey there! How are you?")

=======Claude Wants To Call The send_text_to_user Tool=======
TEXT MESSAGE SENT: Hello! I'm doing well, thanks for asking. How can I assist you today?


Claude responds atrás by calling the `send_text_to_user` tool.

próximo, we'll ask Claude something a bit trickier:

In [164]:
sms_chatbot("I need help looking up an order")

=======Claude Wants To Call The send_text_to_user Tool=======
TEXT MESSAGE SENT: Hi there, to look up your order details I'll need your username first. Can you please provide me with your username?


Claude wants para enviar a texto message, asking a usuário para provide their username.

agora, let's see o que happens quando we provide Claude com our username:

In [165]:
sms_chatbot("I need help looking up an order.  My username is jenny76")

=======Claude Wants To Call The get_customer_info Tool=======
{'username': 'jenny76', 'email': 'jenny76@email.com', 'purchases': [{'id': 1, 'product': 'computer mouse'}, {'id': 2, 'product': 'screen protector'}, {'id': 3, 'product': 'usb charging cable'}]}


Claude calls the `get_customer_info` tool, just as we hoped! 

Even se we enviar Claude a gibberish message, it will still call one of our tools:

In [166]:
sms_chatbot("askdj aksjdh asjkdbhas kjdhas 1+1 ajsdh")

=======Claude Wants To Call The send_text_to_user Tool=======
TEXT MESSAGE SENT: I'm afraid I didn't understand your query. Could you please rephrase what you need help with?
